In [ ]:
from langgraph.graph import StateGraph ,END, START , MessagesState
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI()

In [ ]:
def call_node(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages" : [response]}

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("call_model" , call_node)

builder.add_edge(START , "call_model")
builder.add_edge("call_model" , END)

graph = builder.compile()

graph

In [ ]:
graph.invoke({"messages" : [{"role" : "user" , "content" : "What is the capital of France?"}]})

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})

In [ ]:
def call_model(state: MessagesState):

    response = model.invoke(state["messages"])

    return {"messages": [response]}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable":{"thread_id": "thread-1"}}
config2 = {"configurable":{"thread_id": "thread-2"}}

In [ ]:
graph.invoke({"messages" : [{"role" : "user" , "content" : "What is the capital of France?"}]} , config1)

In [ ]:
snap = graph.get_state(config1)
vals = snap.values
for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content)

In [ ]:
snap = graph.get_state(config2)
vals = snap.values
for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content)